## Tutorial: Replicating the PERLA post-processing using the tranform library.

In [57]:
import json
import pprint
from copy import deepcopy

from deepdiff import DeepDiff
from nomad.datamodel.metainfo.annotations import Rules

from nomad_llm_extraction.transform.inplace_transformer import InplaceTransformer
from nomad_llm_extraction.transform.json_transformer import (
    ProcessingPipeline,
    get_paths,
    update_data,
)
from nomad_llm_extraction.transform.utils import (
    clean_pydantic_jsonschema,
    deref,
    get_cut_data,
    resolve_schema,
)
from nomad_llm_extraction.utils.utils import convert_to_nomad_unit

### Some helper functions for vizualising the processing step differences

In [58]:
def format_deepdiff_example(changes, change_type=None):
    """Format the first example reported for one DeepDiff change type."""
    if (
        change_type == 'dictionary_item_added'
        or change_type == 'dictionary_item_removed'
    ):
        return str(next(iter(changes)))
    if isinstance(changes, dict):
        path = next(iter(changes))
        details = changes[path]

        if isinstance(details, dict):
            old_value = details.get('old_value', details.get('old_type'))
            new_value = details.get('new_value', details.get('new_type'))
            if old_value is not None or new_value is not None:
                return f'{path}: {old_value} -> {new_value}'

        return f'{path}: {details}'

    if isinstance(changes, (list, tuple, set)):
        return str(next(iter(changes)))

    return changes


def print_archive_changes(step_name, before_archive, after_archive, max_changes=6):
    """Print one DeepDiff example for each reported archive change type."""
    diff = DeepDiff(before_archive, after_archive).to_dict()

    print(f'\nProccessing step: {step_name}')
    if not diff:
        print('  No archive changes detected.')
        return

    # print(f'  DeepDiff change types: {len(diff)}')
    for change_type, changes in list(diff.items())[:max_changes]:
        print(f'  {change_type}:')
        print(f'    {format_deepdiff_example(changes, change_type)}')

### Loading the schemas and the data to be processed
The schema exported by the NOMAD metainfo definition is our target schema that the transformed archive should conform to.
The resolve_schema function populates the refs from the \\$defs field.
The processing pipeline uses the [Scalpl](https://github.com/ducdetronquito/scalpl) package internally which allows for path based access to dicts by converting the dict into a `Cut` object which is just a wrapper class. This requires that the \\$defs field be removed after resolving if the schema uses URIs with `.` in its ids. This is only needed if you want to also use the pipeline to modify the schema itself.

The Perla LLM extraction schema is loaded from file and resolved. This is the schema that describes the structure of the extracted archive data before transformation. The schema was generated from the pydantic model use dfor the data extraction. Since it was generated from pydantic we run an extra clean-up step to remove artifact sections like `{type: 'null'}` that pydantic generates for optional fields.

The loaded archive data is a list of json objects i.e you can run the pipelines on a list of archives


In [59]:
with open('tests/data/nomad_llm_extraction_schema.json') as schema_file:
    schema = json.load(schema_file)
resolved_target_schema = resolve_schema(schema, remove_defs=True)

with open('tests/data/llm_extraction_schema.json') as schema_file:
    schema = json.load(schema_file)
resolved_source_schema = clean_pydantic_jsonschema(resolve_schema(schema))

# Load example archive data used as input for the tutorial.
with open('tests/data/10.1002--aenm.202506634.json') as archive_file:
    archive = json.load(archive_file)['cells']

In [60]:
# The archive contains data whose units have already been normalized so we change the value and unit of one of the fields for demo purposes.
print(archive[0]['perovskite_composition']['bandgap'])
archive[0]['perovskite_composition']['bandgap']['value'] = (
    archive[0]['perovskite_composition']['bandgap']['value'] * 1.60218e-19
)
archive[0]['perovskite_composition']['bandgap']['unit'] = 'joule'
print(archive[0]['perovskite_composition']['bandgap'])

{'value': 1.77, 'unit': 'eV'}
{'value': 2.8358586e-19, 'unit': 'joule'}


-----------------------------------------------------------------------------
Processing Step
-----------------------------------------------------------------------------

Each proccesing step in a `ProcessingPipeline` uses a transformation function, a condition fucntion, and an argument-builder function. 

### Condition Function
Function Signature: func_name(section,state) -> bool
The condition functiondecides whether the transformation should run for a given schema section. Condition functions take the current schema section and traversal state as input and return a boolean. 


The state contains path metadata about the current position in the schema:
- 'sname', # the full path of the current section with section names in the schema
- 'name', # the name of the current section
- 'p_name', # the name of the parent section
- 'p_path', # the path of the parent section in the schema
- 'a_p_path', # the path of the parent section in the json object
- 'path', # the path of the current section in the schema
- 'a_path', # the path of the current section in the json object
- 'type', # the type of the current section (property, array_property, section_property)

If None is passed then the transformation will be applied to all the paths.

### Argument Function
Function Signature: func_name(section,state) -> state, args

Argument-builder functions take the same input but return a tuple of the updated state and the arguments to be passed to the transformation function.
This allows the arguments to be dynamically built based on the current position in the schema and modify the paths in the state that are used for later transformations. The returned args should a single object.

Array paths in the state are represented with `n` placeholders such as `layers[n].deposition[n].solution.solutes[n].concentration`.

When applying the pipeline we pass the type of the object being proccesed to be specified, either an `archive` i.e a json object or `schema` i.e the json schema itself.

### Transformation Function
Function Signature: func_name(cut_object,path,args) -> cut_object

The transformation functions take the archive as a Cut object, the path to field on which to apply the transformation and the args that were returned by the arguement builder for that particular path. Since the entire archive is the input and not just the particular section you can have transformations that move data around (e.g the split_value_unit function shown later)

### Example Processing Step Rename Fields

In [61]:
"""`KEY_MAPPING` maps names from one schema vocabulary to another.
The reverse mapping is used while renaming sections in archive data.

Example:
- `bandgap` in one representation becomes `band_gap` in the other.
"""
KEY_MAPPING = {
    'bandgap': 'band_gap',
    'PCE_at_the_start_of_the_experiment': 'PCE_at_start',
    'PCE_at_the_end_of_experiment': 'PCE_at_end',
    'a_ions': 'ions_a_site',
    'b_ions': 'ions_b_site',
    'x_ions': 'ions_x_site',
    'time': 'time',
}
REV_KEY_MAPPING = {value: key for key, value in KEY_MAPPING.items()}


# Condition Function
def rename_cond(section, state):
    return state['name'] in REV_KEY_MAPPING


# Argument Builder Function
def rename_args(section, state):
    """Build the source and destination paths used for archive renaming using the target schema

    The pipeline stores the original archive path in `a_path` and the current
    field name in `name`. We compute the archive path that should exist after the
    rename using the parent archive path 'a_p_path' and return both the updated state and the original path.
    """
    state['a_path'] = f'{state["a_p_path"]}.{REV_KEY_MAPPING[state["name"]]}'
    return state, f'{state["a_p_path"]}.{state["name"]}'


# Transformation Function
def rename_section(jbobj, path, new_path):
    """Rename a field in the archive while preserving its value.

    A deep copy is used so the move stays explicit and safe even when the stored
    value is nested.
    """
    temp_value = deepcopy(jbobj[path])
    jbobj[new_path] = temp_value
    del jbobj[path]
    return jbobj

In [62]:
# Here is a pipeline with a single processing step to rename the fields in the archive
rename_pipeline = ProcessingPipeline(
    {
        'rename': [
            rename_section,  # Transformation Function
            rename_cond,  # Condition Function
            rename_args,  # Argument Builder Function
        ]
    }
)
# Calling the pipeline
renamed_archive = rename_pipeline.apply(data=archive, schema=resolved_target_schema)

2026-04-07 14:03:23.648 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: rename


In [63]:
# Check how the field names change in the archives
print('band_gap' in archive[0]['perovskite_composition'])
print('bandgap' in renamed_archive[0]['perovskite_composition'])
print(
    renamed_archive[0]['perovskite_composition']['band_gap']
    == archive[0]['perovskite_composition']['bandgap']
)

False
False
True


# What happens when the apply function is called
Internally the pipeline uses two functions to perform the processing step `get_paths` and  `update_data`.

Within the pipeline the archive first gets converted to a Cut object which allows for path based access 
e.g instead of archive[0]['perovskite_composition']['bandgap'] we can instead use c_archive[0]['perovskite_composition.bandgap'].

get_paths takes the schema, condition function and the argument builder function and returns all the paths that satisfy the condition with the args needed by the transformation function. the a_path signifies that the transformation function would then operate on the archive, for schemas pass 'path' instead.

In the code the c_ prefix is used to signify that we are operating on a Cut version of the dict.

In [64]:
c_archive = get_cut_data(archive)

rename_paths = get_paths(resolved_target_schema, rename_cond, rename_args, 'a_path')

For each path we have:
- Path state
- Args to be sent to the transformation section, here it is the new path
- Type of the field in the archive, either a 'property' i.e could be avalue or a dict or an 'array' i.e list where the function then has to be applied on a each instance

In [65]:
# State
pprint.pp(rename_paths['perovskite_composition.bandgap'][0])

# Args
print(rename_paths['perovskite_composition.bandgap'][1])

# Type
print(rename_paths['perovskite_composition.bandgap'][2])
print(rename_paths['perovskite_composition.b_ions[n]'][2])

{'sname': 'LLMExtractedPerovskiteSolarCell.perovskite_composition.bandgap',
 'name': 'band_gap',
 'p_name': 'perovskite_composition',
 'p_path': 'properties.perovskite_composition.allOf[1].properties',
 'a_p_path': 'perovskite_composition',
 'path': 'properties.perovskite_composition.allOf[1].properties.band_gap',
 'a_path': 'perovskite_composition.bandgap',
 'type': 'property'}
perovskite_composition.band_gap
property
array


The update_data function then takes the returned paths and uses them to apply the transformation function

In [66]:
renamed_archive_2 = update_data(c_archive, rename_paths, rename_section)

In [67]:
print('band_gap' in archive[0]['perovskite_composition'])
print('bandgap' in renamed_archive_2[0]['perovskite_composition'])
print(
    renamed_archive_2[0]['perovskite_composition.band_gap']
    == archive[0]['perovskite_composition']['bandgap']
)

False
False
True


In [68]:
# The renamed_archive_2 is still a Cut object to convert it back to normal dict use the deref function
print(type(renamed_archive[0]) == type(renamed_archive_2[0]))
print(type(renamed_archive[0]) == type(deref(renamed_archive_2[0])))

False
True


If preferred one could combine the three functions into one transformation function that would then just be applied to all the paths, but for this we would instead use the source schema to get the paths. 

In [69]:
# Transformation Function
def rename_section_source(jbobj, path, func_args):
    """Rename a field in the archive while preserving its value.

    A deep copy is used so the move stays explicit and safe even when the stored
    value is nested.
    """
    name = path.split('.')[-1]
    parent_path = '.'.join(path.split('.')[:-1])
    if name in KEY_MAPPING:
        temp_value = deepcopy(jbobj[path])
        new_path = f'{parent_path}.{KEY_MAPPING[name]}'
        jbobj[new_path] = temp_value
        del jbobj[path]
    return jbobj


rename_pipeline = ProcessingPipeline(
    {
        'rename': [
            rename_section_source,  # Transformation Function
            None,  # Condition Function
            None,
        ]
    }
)  # Argument Builder Function
# Calling the pipeline
renamed_archive_source = rename_pipeline.apply(
    data=archive, schema=resolved_source_schema
)
print('bandgap' in renamed_archive_source[0]['perovskite_composition'])
print(
    renamed_archive_source[0]['perovskite_composition']['band_gap']
    == archive[0]['perovskite_composition']['bandgap']
)

2026-04-07 14:03:25.815 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: rename


False
True


### Full Pipeline
Here are some of the other functions for the processing steps

In [70]:
# Convert units to Nomad Units
def unit_cond(section, state):
    return 'unit' in section


def unit_args(section, state):
    return state, section['unit']


def convert_unit(jbobj, path, unit):
    """Convert archive values to the target NOMAD unit.

    The archive stores some entries as dictionaries such as
    `{"value": 10, "unit": "mg/mL"}`. This helper converts those values into
    the canonical unit declared by the schema.

    It supports both single values and lists of values.
    """
    if jbobj[path] is None:
        return jbobj

    items = jbobj[path]
    if isinstance(items, list):
        items = [
            convert_to_nomad_unit(item['value'], item['unit'], unit) for item in items
        ]
    else:
        items = convert_to_nomad_unit(items['value'], items['unit'], unit)

    jbobj[path] = items
    return jbobj


# Fields listed here are removed completely from the output archive.
SKIP_KEYS = ['additives']


def delete_cond(section, state):
    """Return `True` for fields that should be removed from the archive."""
    return state['name'] in SKIP_KEYS


def delete_section(jbobj, path, func_args):
    """Delete a field from the archive."""
    del jbobj[path]
    return jbobj


# Fields listed here are stored as `{"value": ..., "unit": ...}` and should be split \
# into two separate archive fields such as `concentration` and `concentration_unit`.

SPLIT_VALUE_UNIT = ['concentration']


def split_value_unit_cond(section, state):
    """Return `True` for fields that should be split into value and unit."""
    return state['name'] in SPLIT_VALUE_UNIT


def split_value_unit(jbobj, path, func_args):
    """Split a `{value, unit}` entry into separate archive fields.

    Example:
        `concentration = {"value": 3, "unit": "mol/L"}`

    becomes:
        `concentration = 3`
        `concentration_unit = "mol/L"`
    """
    if jbobj[path] is None:
        return jbobj

    value = jbobj[path]['value']
    unit = jbobj[path]['unit']
    jbobj[path] = value
    jbobj[f'{path}_unit'] = unit
    return jbobj


# Function to create new field in the archive with all the layers names combined into a list
def layer_order__cond(section, state):
    """Return `True` when the current field contains the ordered layer stack."""
    return state['name'] == 'layers'


def layer_order_args(section, state):
    """Provide the archive path used to derive a flattened layer order string."""
    return state, state['a_p_path']


def get_layer_order(layers):
    """Return a comma-separated representation of the device layer stack.

    Missing or malformed layer data returns `None`, which keeps the downstream
    logic simple and explicit.
    """
    if not layers or not isinstance(layers, list):
        return None

    names = [layer['name'] for layer in layers if layer.get('name')]
    return ','.join(names)


def layer_order(jbobj, path, func_args):
    """Create a convenience field called `layer_order` from the `layers` list."""
    jbobj['layer_order'] = get_layer_order(jbobj[path])
    return jbobj


# Flattens the unit value dict to just the value to match the format NOMAD expects it
def remove_unit_value(jbobj, path, func_args):
    """Strip `value` wrappers after unit conversion is complete.

    Example:
        `{"value": 1.2, "unit": "eV"}` becomes `1.2`

    The list case is handled because some archive paths may contain repeated
    entries.
    """
    if isinstance(jbobj[path], list):
        jbobj[path] = [
            item['value'] if (isinstance(item, dict) and 'value' in item) else item
            for item in jbobj[path]
        ]
    elif isinstance(jbobj[path], dict) and 'value' in jbobj[path]:
        jbobj[path] = jbobj[path]['value']
    return jbobj


# Removes Null value fields
def remove_none(jbobj, path, func_args):
    """Remove archive entries whose value is `None`."""
    if jbobj[path] is None:
        del jbobj[path]
    return jbobj

In [71]:
# Passing None for the condition function results in the tranformation function being applied to all possible paths
processing_steps = {
    'rename': [rename_section, rename_cond, rename_args],
    'unit_conversion': [convert_unit, unit_cond, unit_args],
    'split_unit_value': [
        split_value_unit,
        split_value_unit_cond,
        None,
    ],  # None passed for the Argument Builder as we dont need it
    'flatten_unit_value': [
        remove_unit_value,
        None,
        None,
    ],  # None passed for the condition function as we combine the condition within the remove_unit_value itself.
    'delete_sections': [delete_section, delete_cond, None],
    'layer_order': [layer_order, layer_order__cond, layer_order_args],
    'remove_none': [remove_none, None, None],
}

### Single-step transformations demonstrations 
To get a bettter understanding of whats happening at each processing step we will first apply each one of them individually and see what changed using DeepDiff

In [72]:
single_step_archive = deepcopy(archive)
for step_name, step in processing_steps.items():
    single_step_pipeline = ProcessingPipeline({step_name: step})
    next_single_step_archive = single_step_pipeline.apply(
        deepcopy(single_step_archive), resolved_target_schema
    )
    print_archive_changes(step_name, single_step_archive, next_single_step_archive)
    single_step_archive = next_single_step_archive

2026-04-07 14:03:26.807 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: rename
2026-04-07 14:03:26.852 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: unit_conversion
2026-04-07 14:03:26.910 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: split_unit_value
2026-04-07 14:03:26.962 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: flatten_unit_value
2026-04-07 14:03:27.025 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: delete_sections



Proccessing step: rename
  dictionary_item_added:
    root[0]['perovskite_composition']['band_gap']
  dictionary_item_removed:
    root[0]['perovskite_composition']['a_ions']

Proccessing step: unit_conversion
  values_changed:
    root[0]['perovskite_composition']['band_gap']['value']: 2.8358586e-19 -> 1.7700037185787596

Proccessing step: split_unit_value
  type_changes:
    root[0]['layers'][2]['deposition'][0]['solution']['solutes'][0]['concentration']: {'value': 0.001, 'unit': 'mol/L'} -> 0.001
  dictionary_item_added:
    root[0]['layers'][2]['deposition'][0]['solution']['solutes'][0]['concentration_unit']

Proccessing step: flatten_unit_value
  type_changes:
    root[0]['perovskite_composition']['band_gap']: {'value': 1.7700037185787596, 'unit': 'electron_volt'} -> 1.7700037185787596


2026-04-07 14:03:27.062 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: layer_order
2026-04-07 14:03:27.107 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: remove_none



Proccessing step: delete_sections
  dictionary_item_removed:
    root[0]['perovskite_composition']['additives']

Proccessing step: layer_order
  dictionary_item_added:
    root[0]['layer_order']

Proccessing step: remove_none
  dictionary_item_removed:
    root[0]['number_devices']
  values_changed:
    root[0]['layers'][4]['deposition'][0]: {'step_name': None, 'method': 'Evaporation', 'atmosphere': 'Vacuum', 'temperature': None, 'duration': None, 'antisolvent': None, 'solution': None, 'additional_parameters': None} -> {'method': 'Evaporation', 'atmosphere': 'Vacuum'}


In practice, we can compose them into one single pipeline and run them in sequence.

In [73]:
processing_pipeline = ProcessingPipeline(processing_steps)

# Apply the pipeline to the archive data .
updated_archive = processing_pipeline.apply(deepcopy(archive), resolved_target_schema)
assert single_step_archive == updated_archive

2026-04-07 14:03:27.179 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: rename
2026-04-07 14:03:27.184 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 1: unit_conversion
2026-04-07 14:03:27.193 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 2: split_unit_value
2026-04-07 14:03:27.197 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 3: flatten_unit_value
2026-04-07 14:03:27.219 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 4: delete_sections
2026-04-07 14:03:27.222 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 5: layer_order
2026-04-07 14:03:27.225 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 6: remove_none


The processing pipeline has an additional functionality of cleaning up the archive by removing the paths that are not present in the target_schema. You can also pass in a custom function to it that would then be applied to these paths (default behavior is deleting the field). The function uses the source_schema of the json object to get the paths that might be present within the json object. 

If no source schema is provided then all fields that are in the json object but not in the target schema are removed. If no target schema is provided then the behavior is similar to the remove none function but additionally fields having empty strings,lists,tuples,dicts values are also removed.

In [74]:
cleaned_archive = processing_pipeline.clean(
    updated_archive, resolved_target_schema, resolved_source_schema
)

In [75]:
DeepDiff(updated_archive, cleaned_archive)

{'dictionary_item_removed': ["root[0]['layers'][2]['deposition'][0]['solution']['compounds']", "root[0]['layers'][3]['deposition'][0]['solution']['compounds']", "root[1]['layers'][2]['deposition'][0]['solution']['compounds']", "root[1]['layers'][3]['deposition'][0]['solution']['compounds']", "root[2]['layers'][2]['deposition'][0]['solution']['compounds']", "root[2]['layers'][3]['deposition'][0]['solution']['compounds']", "root[2]['layers'][5]['deposition'][0]['solution']['compounds']", "root[3]['layers'][3]['deposition'][0]['solution']['compounds']"]}

# Modifying the Schema
We can also use the pipeline to modify the schema. In the pipeline we then need to have the process type as 'schema'

In [76]:
def unit_cond(section, state):
    return 'unit' in section


def unit_args(section, state):
    return state, section['unit']


def update_unit_value_schema(jsobj, path, unit):
    """Expand a schema field into a `value`/`unit` object schema.

    This temporarily converts a flat schema property with an unit associated with it into an object like:

    {
        "properties": {
            "value": <original field schema>,
            "unit": {"type": "string", "enum": [unit]}
        }
    }
    """
    value_schema = deepcopy(jsobj[path])
    del value_schema['unit']

    jsobj[path] = {
        'properties': {
            'value': value_schema.copy(),
            'unit': {'type': 'string', 'enum': [unit]},
        }
    }

    for key in ['title', 'description']:
        if key in value_schema:
            jsobj[path][key] = value_schema[key]

    return jsobj

In [77]:
# Reversve transformation of the above function
def flatten_unit_value_schema_cond(section, state):
    """Identify schema sections shaped like `{properties: {value, unit}}`.

    These are the sections we temporarily expanded during schema transformation and then
    flattened back after validation.
    """
    return (
        'properties' in section
        and 'value' in section['properties']
        and 'unit' in section['properties']
    )


def flatten_unit_value_schema(jsobj, path, func_args):
    """Collapse a temporary `value`/`unit` schema back into a flat form."""
    value_schema = deepcopy(jsobj[path])
    jsobj[path].update(value_schema['properties']['value'])
    jsobj[path]['unit'] = value_schema['properties']['unit']['enum'][0]
    del jsobj[path]['properties']
    return jsobj

In [78]:
schema_pipeline = ProcessingPipeline(
    {
        'add_unit_value': [update_unit_value_schema, unit_cond, unit_args]
    }  # the condition and arg builder are the same as in the unit conversion step
)

updated_schema = schema_pipeline.apply(resolved_target_schema, proc_type='schema')
print_archive_changes('schema_unit_expansion', resolved_target_schema, updated_schema)

2026-04-07 14:03:28.551 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: add_unit_value



Proccessing step: schema_unit_expansion
  values_changed:
    root['properties']['jsc']: {'description': 'Short-circuit current density (JSC)', '$id': 'https://nomad-lab.eu/prod/v1/api/v1/models/perovskite_solar_cell_database.llm_extraction_schema.LLMExtractedPerovskiteSolarCell.jsc@8ff4eed103fdefd080d86b44a71e913153cebd0c', 'type': 'number', 'unit': 'milliampere / centimeter ** 2'} -> {'properties': {'value': {'description': 'Short-circuit current density (JSC)', '$id': 'https://nomad-lab.eu/prod/v1/api/v1/models/perovskite_solar_cell_database.llm_extraction_schema.LLMExtractedPerovskiteSolarCell.jsc@8ff4eed103fdefd080d86b44a71e913153cebd0c', 'type': 'number'}, 'unit': {'type': 'string', 'enum': ['milliampere / centimeter ** 2']}}, 'description': 'Short-circuit current density (JSC)'}


In [79]:
print('Old Schema Section:')
pprint.pp(resolved_target_schema['properties']['voc'])
print('New Schema Section:')
pprint.pp(updated_schema['properties']['voc'])

Old Schema Section:
{'description': 'Open-circuit voltage (VOC)',
 '$id': 'https://nomad-lab.eu/prod/v1/api/v1/models/perovskite_solar_cell_database.llm_extraction_schema.LLMExtractedPerovskiteSolarCell.voc@184a39d9b08ac04f1e8a80822a673cee7840b92a',
 'type': 'number',
 'unit': 'volt',
 'minimum': 0}
New Schema Section:
{'properties': {'value': {'description': 'Open-circuit voltage (VOC)',
                          '$id': 'https://nomad-lab.eu/prod/v1/api/v1/models/perovskite_solar_cell_database.llm_extraction_schema.LLMExtractedPerovskiteSolarCell.voc@184a39d9b08ac04f1e8a80822a673cee7840b92a',
                          'type': 'number',
                          'minimum': 0},
                'unit': {'type': 'string', 'enum': ['volt']}},
 'description': 'Open-circuit voltage (VOC)'}


In [80]:
# reversing the change to the schema we made. The end DeepDiff should show no changes between the original and updated_schema
rev_schema_pipeline = ProcessingPipeline(
    {
        'remove_unit_value': [
            flatten_unit_value_schema,
            flatten_unit_value_schema_cond,
            None,
        ]
    }
)
updated_schema = rev_schema_pipeline.apply(
    updated_schema, proc_type='schema'
)  # We pass the expanded schema here
print_archive_changes(
    'schema_reverse_unit_expansion', resolved_target_schema, updated_schema
)  # The schema should now match the original schema again

2026-04-07 14:03:28.925 | INFO     | nomad_llm_extraction.transform.json_transformer:apply:241 - Applying processing step 0: remove_unit_value



Proccessing step: schema_reverse_unit_expansion
  No archive changes detected.


## Inplace Transformer
The transform module also comes with an inplace extension of the NOMAD json Transformer ([docs](https://nomad-lab.eu/prod/v1/docs/howto/manage/program/json_transformer.html)) with array handling. The json transformer i sprimarily to move/restructure the data. 

We will replicate some of the transformations we did above using the InplaceTransformer. 

In [81]:
rules = {
    'transforms': Rules(
        name='',
        rules={
            'split_concentration_map': {
                'source': 'layers[n1].deposition[n2].solution.solutes[n3].concentration.value',
                'target': 'layers[n1].deposition[n2].solution.solutes[n3].concentration',
            },
            'split_concentration_map2': {
                'source': 'layers[n1].deposition[n2].solution.solutes[n3].concentration.unit',
                'target': 'layers[n1].deposition[n2].solution.solutes[n3].concentration_unit',
            },
            'rename_0': {
                'source': 'perovskite_composition.bandgap',
                'target': 'perovskite_composition.band_gap',
            },
            'rename_1': {
                'source': 'perovskite_composition.a_ions',
                'target': 'perovskite_composition.ions_a_site',
            },
            'rename_2': {
                'source': 'perovskite_composition.b_ions',
                'target': 'perovskite_composition.ions_b_site',
            },
            'rename_3': {
                'source': 'perovskite_composition.x_ions',
                'target': 'perovskite_composition.ions_x_site',
            },
            'rename_4': {'source': 'stability.time', 'target': 'stability.time'},
            'rename_5': {
                'source': 'stability.PCE_at_the_start_of_the_experiment',
                'target': 'stability.PCE_at_start',
            },
            'rename_6': {
                'source': 'stability.PCE_at_the_end_of_experiment',
                'target': 'stability.PCE_at_end',
            },
        },
    )
}

In [82]:
transformer = InplaceTransformer(rules)
updated_archive = transformer.transform_inplace(archive, 'transforms')
print_archive_changes('Inplace', archive, updated_archive)


Proccessing step: Inplace
  type_changes:
    root[0]['layers'][2]['deposition'][0]['solution']['solutes'][0]['concentration']: {'value': 0.001, 'unit': 'mol/L'} -> 0.001
  dictionary_item_added:
    root[0]['perovskite_composition']['band_gap']
  dictionary_item_removed:
    root[0]['perovskite_composition']['a_ions']
